# চ্যালেঞ্জ: ডেটা সায়েন্স সম্পর্কে টেক্সট বিশ্লেষণ

এই উদাহরণে, চলুন একটি সাধারণ অনুশীলন করি যা ঐতিহ্যবাহী ডেটা সায়েন্স প্রক্রিয়ার সকল ধাপ আবৃত করে। আপনাকে কোনো কোড লিখতে হবে না, আপনি শুধু নিচের সেল গুলোতে ক্লিক করে তা চালাতে পারেন এবং ফলাফল পর্যবেক্ষণ করতে পারেন। একটি চ্যালেঞ্জ হিসেবে, আপনাকে বিভিন্ন ডেটার সাথে এই কোডটি চেষ্টা করার জন্য উৎসাহিত করা হয়েছে।

## লক্ষ্য

এই পাঠে, আমরা ডেটা সায়েন্স সম্পর্কিত বিভিন্ন ধারণা আলোচনা করছি। চলুন কিছু **টেক্সট মাইনিং** করে আরও সম্পর্কিত ধারণা আবিষ্কার করার চেষ্টা করি। আমরা ডেটা সায়েন্স সম্পর্কে একটি টেক্সট থেকে শুরু করব, তাতে থেকে কীওয়ার্ড বের করব এবং তারপর ফলাফল ভিজ্যুয়ালাইজ করার চেষ্টা করব।

একটি টেক্সট হিসেবে, আমি উইকিপিডিয়ার ডেটা সায়েন্স পৃষ্ঠা ব্যবহার করব:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## ধাপ ১: ডেটা সংগ্রহ করা

প্রতিটি ডেটা সায়েন্স প্রক্রিয়ার প্রথম ধাপ হল ডেটা সংগ্রহ করা। আমরা এটার জন্য `requests` লাইব্রেরি ব্যবহার করব:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## ধাপ ২: ডেটা রূপান্তর করা

পরবর্তী ধাপ হল প্রক্রিয়াকরণের জন্য ডেটাকে উপযুক্ত ফর্মে রূপান্তর করা। আমাদের ক্ষেত্রে, আমরা পেজ থেকে HTML সোর্স কোড ডাউনলোড করেছি এবং এটাকে প্লেইন টেক্সটে রূপান্তর করতে হবে।

এটি করার অনেক উপায় আছে। আমরা ব্যবহার করব [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), যা একটি জনপ্রিয় পাইথন লাইব্রেরি HTML পার্স করার জন্য। BeautifulSoup আমাদের নির্দিষ্ট HTML উপাদানগুলোতে লক্ষ্য করার সুযোগ দেয়, তাই আমরা উইকিপিডিয়ার মূল আর্টিকেল বিষয়বস্তুর উপর কেন্দ্রীভূত হতে পারব এবং কিছু ন্যাভিগেশন মেনু, সাইডবার, ফুটার, এবং অন্যান্য প্রাসঙ্গিক নয় এমন উপাদান কমিয়ে আনতে পারব (যদিও কিছু বোলারপ্লেট টেক্সট এখনও থাকতে পারে)।


প্রথমে, আমাদের HTML পার্সিংয়ের জন্য BeautifulSoup লাইব্রেরি ইনস্টল করতে হবে:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## ধাপ ৩: অন্তর্দৃষ্টি গ্রহণ

সবচেয়ে গুরুত্বপূর্ণ ধাপ হল আমাদের ডেটাকে এমন একটি রূপে পরিণত করা যেটি থেকে আমরা অন্তর্দৃষ্টি অনুধাবন করতে পারি। আমাদের ক্ষেত্রে, আমরা টেক্সট থেকে কীওয়ার্ড বের করতে চাই, এবং দেখতে চাই কোন কীওয়ার্ডগুলি বেশি অর্থবহ।

আমরা কীওয়ার্ড এক্সট্রাকশনের জন্য Python লাইব্রেরি [RAKE](https://github.com/aneesha/RAKE) ব্যবহার করব। প্রথমে, ধরে নেই লাইব্রেরিটি না থাকলে এটি ইনস্টল করি: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

প্রধান কার্যকারিতা `Rake` অবজেক্ট থেকে পাওয়া যায়, যাকে আমরা কিছু প্যারামিটার ব্যবহার করে কাস্টমাইজ করতে পারি। আমাদের ক্ষেত্রে, আমরা একটি কীওয়ার্ডের সর্বনিম্ন দৈর্ঘ্য ৫টি অক্ষর, ডকুমেন্টে একটি কীওয়ার্ডের সর্বনিম্ন ফ্রিকোয়েন্সি ৩, এবং একটি কীওয়ার্ডে সর্বোচ্চ শব্দের সংখ্যা ২ সেট করব। অন্য মানগুলোর সাথে পরীক্ষা চালাতে পারেন এবং ফলাফল পর্যবেক্ষণ করতে পারেন।


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


আমরা গুরুত্বের সাথে সম্পর্কিত একটি শর্তের তালিকা পেয়েছি। যেমন আপনি দেখতে পাচ্ছেন, সবচেয়ে সংশ্লিষ্ট বিষয়গুলি, যেমন মেশিন লার্নিং এবং বিগ ডেটা, তালিকায় শীর্ষ অবস্থানে উপস্থিত রয়েছে।

## ধাপ ৪: ফলাফল চিত্রায়ন করা

মানুষ ডেটাকে ভিজ্যুয়াল আকারে সবচেয়ে ভালভাবে ব্যাখ্যা করতে পারে। তাই কিছু অন্তর্দৃষ্টি আঁকতে ডেটা ভিজ্যুয়ালাইজ করা প্রায়ই যুক্তিযুক্ত হয়। আমরা পাইটনে `matplotlib` লাইব্রেরি ব্যবহার করে কীওয়ার্ডগুলির সংশ্লিষ্টতা সহ সরল বণ্টন প্লট করতে পারি:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

তবে, শব্দের ফ্রিকোয়েন্সি ভিজ্যুয়ালাইজ করার আরও ভালো একটি উপায় রয়েছে - **ওয়ার্ড ক্লাউড** ব্যবহার করা। আমাদের কীওয়ার্ড তালিকা থেকে ওয়ার্ড ক্লাউড প্লট করতে আরও একটি লাইব্রেরি ইনস্টল করতে হবে।


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` অবজেক্ট মূল টেক্সট বা আগেই গণনা করা শব্দ-তালিকা ও তাদের ফ্রিকোয়েন্সি গ্রহণ করার জন্য এবং একটি ইমেজ রিটার্ন করার জন্য দায়ী, যা পরে `matplotlib` ব্যবহার করে প্রদর্শন করা যেতে পারে:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

আমরা `WordCloud` এ আসল টেক্সটও পাঠাতে পারি - চলুন দেখি আমরা একই রকম ফলাফল পেতে পারি কি না:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

আপনি দেখতে পাচ্ছেন যে ওয়ার্ড ক্লাউড এখন আরও ইমপ্রেসিভ দেখাচ্ছে, তবে এতে অনেক শব্দের গোলমালও রয়েছে (যেমন অবাঞ্ছিত শব্দ যেমন `Retrieved on`)। এছাড়াও, আমরা পাচ্ছি কম কীওয়ার্ড যা দুই শব্দের সমন্বয়ে গঠিত, যেমন *data scientist*, বা *computer science*। এর কারণ RAKE অ্যালগরিদম পাঠ থেকে ভালো কীওয়ার্ড নির্বাচন করার ক্ষেত্রে অনেক ভালো কাজ করে। এই উদাহরণটি ডেটা প্রি-প্রসেসিং এবং ক্লিনিংয়ের গুরুত্বকে উপস্থাপন করে, কারণ শেষে পরিষ্কার ছবি আমাদের আরও ভাল সিদ্ধান্ত নিতে সাহায্য করবে।

এই ব্যায়ামে আমরা উইকিপিডিয়া পাঠ থেকে কিছু অর্থ নির্গত করার একটি সহজ প্রক্রিয়া পার হয়েছি, কীওয়ার্ড এবং ওয়ার্ড ক্লাউড আকারে। এই উদাহরণটি বেশ সহজ, তবে এটি একটি ডেটা সায়েন্টিস্ট যখন ডেটা নিয়ে কাজ করে তখন যে সমস্ত সাধারণ ধাপ নেয় তা ভালভাবে প্রদর্শন করে, ডেটা অধিগ্রহণ থেকে শুরু করে ভিজ্যুয়ালাইজেশন পর্যন্ত।

আমাদের কোর্সে আমরা সেই সমস্ত ধাপ বিশদে আলোচনা করব।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
